In [ ]:
import requests
import pandas as pd
from pathlib import Path

In [ ]:
# -----------------------------
# ENDPOINTS
# -----------------------------
endpoints = {
    "income_statement_standardized": "income-statement",
    "balance_sheet_standardized": "balance-sheet-statement",
    "cash_flow_standardized": "cash-flow-statement",

    # Useful for closer-to-SEC/XBRL names
    "income_statement_as_reported": "income-statement-as-reported",
    "balance_sheet_as_reported": "balance-sheet-statement-as-reported",
    "cash_flow_as_reported": "cash-flow-statement-as-reported",
}

In [ ]:
# -----------------------------
# USER INPUTS
# -----------------------------
API_KEY = "YOUR_FMP_API_KEY"
SYMBOL = "AAPL"
PERIOD = "quarter"
LIMIT = 8

SAVE_DIR = Path.home() / "Desktop" / "Trading" / "Data" / "fmp_line_items"
SAVE_DIR.mkdir(parents=True, exist_ok=True)


# -----------------------------
# FMP REQUEST
# -----------------------------
def fmp_get(endpoint, symbol, api_key, period="quarter", limit=8):
    url = f"https://financialmodelingprep.com/api/v3/{endpoint}/{symbol}"
    params = {
        "period": period,
        "limit": limit,
        "apikey": api_key
    }

    r = requests.get(url, params=params, timeout=30)
    r.raise_for_status()
    data = r.json()

    if not data:
        return pd.DataFrame()

    return pd.DataFrame(data)


# -----------------------------
# DISCOVER ALL NON-NULL ACCOUNTS
# -----------------------------
def discover_accounts_across_filings(df, statement_name):
    if df.empty:
        return pd.DataFrame()

    id_cols = {
        "date", "symbol", "reportedCurrency", "cik",
        "fillingDate", "acceptedDate", "calendarYear",
        "period", "link", "finalLink"
    }

    rows = []

    for col in df.columns:
        if col in id_cols:
            continue

        non_null_mask = df[col].notna()

        if non_null_mask.any():
            rows.append({
                "statement": statement_name,
                "account": col,
                "quarters_present_count": int(non_null_mask.sum()),
                "latest_value": df.loc[non_null_mask, col].iloc[-1]
            })

    return pd.DataFrame(rows).sort_values(
        ["statement", "account"]
    ).reset_index(drop=True)





In [12]:
std_accounts = pd.concat([
    discover_accounts_across_filings(income_std, "income_statement_standardized"),
    discover_accounts_across_filings(balance_std, "balance_sheet_standardized"),
    discover_accounts_across_filings(cashflow_std, "cash_flow_standardized"),
])

raw_accounts = pd.concat([
    discover_accounts_across_filings(income_raw, "income_statement_as_reported"),
    discover_accounts_across_filings(balance_raw, "balance_sheet_as_reported"),
    discover_accounts_across_filings(cashflow_raw, "cash_flow_as_reported"),
])



account_inventory = pd.concat([std_accounts, raw_accounts])

NameError: name 'fmp_get' is not defined

In [ ]:
# -----------------------------
# RUN
# -----------------------------
all_account_inventories = []
raw_statements = {}

for statement_name, endpoint in endpoints.items():
    print(f"Pulling {statement_name}...")

    df = fmp_get(
        endpoint=endpoint,
        symbol=SYMBOL,
        api_key=API_KEY,
        period=PERIOD,
        limit=LIMIT
    )

    raw_statements[statement_name] = df

    raw_path = SAVE_DIR / f"{SYMBOL}_{statement_name}_{PERIOD}_{LIMIT}.csv"
    df.to_csv(raw_path, index=False)

    inventory = discover_accounts_across_filings(df, statement_name)
    all_account_inventories.append(inventory)


account_inventory = pd.concat(all_account_inventories, ignore_index=True)

output_path = SAVE_DIR / f"{SYMBOL}_ALL_ACCOUNTS_last_{LIMIT}_{PERIOD}s.csv"
account_inventory.to_csv(output_path, index=False)

print("\nDone.")
print(f"Saved account inventory to:\n{output_path}")

account_inventory

In [ ]:
# -----------------------------
# USER INPUTS
# -----------------------------
API_KEY = "YOUR_FMP_API_KEY"
SYMBOL = "AAPL"
PERIOD = "quarter"
LIMIT = 8

SAVE_DIR = Path.home() / "Desktop" / "Trading" / "Data" / "fmp_quarters"
SAVE_DIR.mkdir(parents=True, exist_ok=True)


# -----------------------------
# FMP REQUEST
# -----------------------------
def fmp_get(endpoint, symbol, api_key, period="quarter", limit=8):
    url = f"https://financialmodelingprep.com/api/v3/{endpoint}/{symbol}"
    params = {
        "period": period,
        "limit": limit,
        "apikey": api_key
    }

    r = requests.get(url, params=params, timeout=30)
    r.raise_for_status()

    data = r.json()
    return pd.DataFrame(data)


# -----------------------------
# PULL STATEMENTS
# -----------------------------
income = fmp_get("income-statement", SYMBOL, API_KEY, PERIOD, LIMIT)
balance = fmp_get("balance-sheet-statement", SYMBOL, API_KEY, PERIOD, LIMIT)
cashflow = fmp_get("cash-flow-statement", SYMBOL, API_KEY, PERIOD, LIMIT)


# -----------------------------
# BASIC CLEANING
# -----------------------------
def clean_statement(df, prefix):
    df = df.copy()

    id_cols = [
        "date", "symbol", "reportedCurrency", "cik",
        "fillingDate", "acceptedDate", "calendarYear",
        "period", "link", "finalLink"
    ]

    keep_id_cols = [c for c in id_cols if c in df.columns]

    value_cols = [c for c in df.columns if c not in id_cols]

    # Prefix statement-specific columns so names do not collide
    rename_map = {c: f"{prefix}_{c}" for c in value_cols}

    df = df[keep_id_cols + value_cols].rename(columns=rename_map)

    return df


income_clean = clean_statement(income, "is")
balance_clean = clean_statement(balance, "bs")
cashflow_clean = clean_statement(cashflow, "cf")


# -----------------------------
# MERGE STATEMENTS BY QUARTER
# -----------------------------
quarterly = (
    income_clean
    .merge(balance_clean, on=["date", "symbol"], how="outer")
    .merge(cashflow_clean, on=["date", "symbol"], how="outer")
)

quarterly["date"] = pd.to_datetime(quarterly["date"])
quarterly = quarterly.sort_values("date").reset_index(drop=True)


# -----------------------------
# CALCULATE YOY % CHANGE
# -----------------------------
def add_yoy_percent_changes(df):
    df = df.copy()

    id_cols = {
        "date", "symbol", "reportedCurrency", "cik",
        "fillingDate", "acceptedDate", "calendarYear",
        "period",
        "link", "finalLink"
    }

    numeric_cols = [
        c for c in df.columns
        if c not in id_cols and pd.api.types.is_numeric_dtype(df[c])
    ]

    for col in numeric_cols:
        prior_year_value = df[col].shift(4)

        df[f"{col}_yoy_pct_change"] = (
            (df[col] - prior_year_value) / prior_year_value.abs()
        )

    return df


quarterly_with_yoy = add_yoy_percent_changes(quarterly)


# -----------------------------
# SAVE OUTPUTS
# -----------------------------
raw_output_path = SAVE_DIR / f"{SYMBOL}_last_{LIMIT}_quarters_raw.csv"
yoy_output_path = SAVE_DIR / f"{SYMBOL}_last_{LIMIT}_quarters_with_yoy.csv"

quarterly.to_csv(raw_output_path, index=False)
quarterly_with_yoy.to_csv(yoy_output_path, index=False)

print("Done.")
print(f"Raw quarterly data saved to:\n{raw_output_path}")
print(f"Quarterly data with YoY % changes saved to:\n{yoy_output_path}")

quarterly_with_yoy

In [ ]:
# -----------------------------
# USER INPUTS
# -----------------------------
TAX_RATE = 0.21

account_map = {
    # raw_column_name: bucket + sign

    "is_operatingIncome": {
        "bucket": "operating_profit",
        "sign": 1
    },

    "is_netIncome": {
        "bucket": "net_income",
        "sign": 1
    },

    "cf_depreciationAndAmortization": {
        "bucket": "non_cash_charges",
        "sign": 1
    },

    "cf_capitalExpenditure": {
        "bucket": "capex",
        "sign": -1
    },

    "bs_accountsReceivable_delta": {
        "bucket": "working_capital",
        "sign": -1
    },

    "bs_inventory_delta": {
        "bucket": "working_capital",
        "sign": -1
    },

    "bs_accountsPayable_delta": {
        "bucket": "working_capital",
        "sign": 1
    },

    "cf_debtRepayment": {
        "bucket": "net_borrowing",
        "sign": -1
    },

    "cf_debtIssued": {
        "bucket": "net_borrowing",
        "sign": 1
    },
}

In [ ]:
def build_fcf_buckets(df, account_map):
    df = df.copy()

    bucket_cols = {}

    for raw_col, config in account_map.items():
        bucket = config["bucket"]
        sign = config["sign"]

        if raw_col not in df.columns:
            print(f"Warning: missing column skipped: {raw_col}")
            continue

        normalized_col = f"normalized_{raw_col}"
        df[normalized_col] = df[raw_col].fillna(0) * sign

        if bucket not in bucket_cols:
            bucket_cols[bucket] = []

        bucket_cols[bucket].append(normalized_col)

    for bucket, cols in bucket_cols.items():
        df[f"bucket_{bucket}"] = df[cols].sum(axis=1)

    return df

In [ ]:
def calculate_fcff_fcfe_from_buckets(df, tax_rate=0.21):
    df = df.copy()

    # Default missing buckets to zero
    required_buckets = [
        "operating_profit",
        "net_income",
        "non_cash_charges",
        "capex",
        "working_capital",
        "net_borrowing"
    ]

    for bucket in required_buckets:
        col = f"bucket_{bucket}"
        if col not in df.columns:
            df[col] = 0

    df["operating_profit_after_tax"] = (
        df["bucket_operating_profit"] * (1 - tax_rate)
    )

    df["fcff"] = (
        df["operating_profit_after_tax"]
        + df["bucket_non_cash_charges"]
        + df["bucket_capex"]
        + df["bucket_working_capital"]
    )

    df["fcfe"] = (
        df["bucket_net_income"]
        + df["bucket_non_cash_charges"]
        + df["bucket_capex"]
        + df["bucket_working_capital"]
        + df["bucket_net_borrowing"]
    )

    return df

In [ ]:
fcf_bucketed = build_fcf_buckets(
    df=quarterly_with_yoy,
    account_map=account_map
)

fcf_results = calculate_fcff_fcfe_from_buckets(
    df=fcf_bucketed,
    tax_rate=0.21
)

fcf_results[
    [
        "date",
        "symbol",
        "bucket_operating_profit",
        "bucket_net_income",
        "bucket_non_cash_charges",
        "bucket_capex",
        "bucket_working_capital",
        "bucket_net_borrowing",
        "fcff",
        "fcfe"
    ]
]